# §3.5.1 검증 — Moshi self/user 오디오 테이블 코사인

`docs/contexts/ARCHITECTURE.md` §3.5.1은 두 셀(cb0=0.501, cb1=0.751)만 제시하고 이렇게 **예측**한다:

> **함의 2**: role 분화는 semantic에 집중된다. acoustic은 self/user가 이미 유사(0.75)하고 semantic이 갈린다(0.50).
>
> **정직한 confound**: acoustic 레벨 1→7로 갈수록 유사도가 **단조 증가**하는지 확인하면 두 해석("역할 무관성" vs "분화 압력 부족")을 가를 수 있다.

이 노트북은 실제 `kmhf/hf-moshiko`에서 **8개 코드북 전부**의 self/user 코사인을 재서:
1. 문서의 두 published 셀(cb0, cb1)이 재현되는지 (방법론 검증)
2. cb1→cb7이 단조 증가인지 감소인지 (문서의 미해결 질문에 대한 답)

를 확인한다. `project_amnesty.utils.warm_start_haan.moshi_audio_tables`를 그대로 쓴다 — warm-start가 실제로 복사해 넣는 바로 그 텐서다.

## 1. 준비

In [ ]:
import os
os.environ.setdefault("HF_HOME", "/data/hf_cache")   # 체크포인트 캐시 위치
import torch
from project_amnesty.utils.warm_start_haan import moshi_audio_tables

MOSHI = "kmhf/hf-moshiko"
K = 8
print("moshi:", MOSHI, "| HF_HOME:", os.environ["HF_HOME"])

## 2. 두 절반의 테이블을 읽는다

Moshi는 `2K`개 오디오 입력 테이블을 갖는다 — `[0,K)`가 self(assistant), `[K,2K)`가 user.
`moshi_audio_tables(side=...)`가 각 절반을 `(audio_vocab_size+1, hidden)` 텐서 K개로 돌려준다.
그중 **행(=코드 임베딩) 단위 코사인**을 self_i vs user_i로 잰다.

In [ ]:
self_tables = moshi_audio_tables(MOSHI, num_codebooks=K, side="self")
user_tables = moshi_audio_tables(MOSHI, num_codebooks=K, side="user")
print("self 테이블", len(self_tables), "개, shape", tuple(self_tables[0].shape))
print("user 테이블", len(user_tables), "개, shape", tuple(user_tables[0].shape))

## 3. 코드북별 self/user 코사인 + 상수-오프셋 설명 분산

- **row cosine**: 각 코드 id의 self 임베딩 벡터와 user 임베딩 벡터 사이 코사인의 평균.
- **상수 오프셋이 설명하는 분산**: `user - self`가 코드 무관한 하나의 방향(평균 차이)으로
  얼마나 설명되는가. 문서 §3.5.1의 "코드 무관 일정 방향이 설명하는 분산"과 같은 정의.

In [ ]:
def row_cosine(a, b):
    a = a.float(); b = b.float()
    return torch.nn.functional.cosine_similarity(a, b, dim=1).mean().item()

def const_offset_share(a, b):
    # (user - self) 의 총분산 중, 평균 차이 벡터(코드 무관 상수 오프셋)가 설명하는 비율.
    d = (b - a).float()
    total = d.var(dim=0, unbiased=False).sum().item() * d.shape[0] + d.mean(dim=0).pow(2).sum().item() * d.shape[0]
    # 더 단순하게: ||mean||^2 / mean(||d||^2)
    mean_sq = d.mean(dim=0).pow(2).sum().item()
    energy = d.pow(2).sum(dim=1).mean().item()
    return mean_sq / energy if energy > 0 else float("nan")

rows = []
for k in range(K):
    cos = row_cosine(self_tables[k], user_tables[k])
    share = const_offset_share(self_tables[k], user_tables[k])
    rows.append((k, cos, share))

print(f"{'cb':>3} {'cosine':>9} {'const-offset share':>20}")
print("-" * 36)
for k, cos, share in rows:
    print(f"{k:>3} {cos:>9.4f} {share*100:>18.2f}%")

## 4. 문서 예측과 대조

In [ ]:
cos = [c for _, c, _ in rows]

# (a) published 셀 재현? 문서: cb0=0.501, cb1=0.751
print("=== 방법론 검증 (published 두 셀 재현) ===")
print(f"  cb0: 측정 {cos[0]:.3f}  vs 문서 0.501  -> {'재현' if abs(cos[0]-0.501)<0.01 else '불일치'}")
print(f"  cb1: 측정 {cos[1]:.3f}  vs 문서 0.751  -> {'재현' if abs(cos[1]-0.751)<0.01 else '불일치'}")

# (b) 문서의 미해결 질문: cb1..cb7 단조 증가인가?
acoustic = cos[1:]
diffs = [acoustic[i+1] - acoustic[i] for i in range(len(acoustic)-1)]
mono_inc = all(d > 0 for d in diffs)
mono_dec = all(d < 0 for d in diffs)
print("\n=== 문서의 미해결 질문: acoustic cb1->cb7 단조성 ===")
print(f"  cb1..cb7 코사인: {[round(x,3) for x in acoustic]}")
print(f"  단조 증가? {mono_inc}   단조 감소? {mono_dec}")

print("\n=== 판정 (문서 §3.5.1 논리 그대로) ===")
if mono_inc:
    print("  단조 증가 -> 문서 함의 2 지지: acoustic이 role-무관 (해석 '역할 무관성')")
elif mono_dec:
    print("  단조 감소 -> 문서 예측과 반대. cb1은 이상치이고 깊은 acoustic은")
    print("     semantic 수준으로 role-분화. 문서 자체 논리상 '분화 압력 부족' 해석을 지지")
    print("     => §6.1/§6.2 의 '공유 시 잃는 role 정보가 적다' 사전 예측의 근거가 약해짐")
else:
    print("  비단조 -> 단순 요약 불가, cb 별로 봐야 함")

## 5. 결론 요약

아래 셀이 문서에 반영할 한 줄 결론을 만든다. (이 노트북은 문서를 직접 고치지 않는다 —
측정만 제공하고, 반영 문구는 사람이 판단한다.)

In [ ]:
print("측정 결과 요약")
print("=" * 44)
print(f"  cb0 (semantic)   cosine {cos[0]:.4f}")
print(f"  cb1..cb7 (acoustic) {[round(x,3) for x in cos[1:]]}")
print(f"  acoustic 평균     {sum(cos[1:])/7:.4f}")
print(f"  cb1 이 이상치인가 (다음보다 큼): {cos[1] > cos[2]}")
print()
if mono_dec:
    print("문서 반영 제안:")
    print("  §6.1/§6.2 사전예측 'acoustic은 이미 유사(0.75)라 공유 손실 적다' 는")
    print("  cb1 단일 셀 일반화였음. 실측상 acoustic 코사인은 cb1=%.2f 에서" % cos[1])
    print("  cb7=%.2f 로 단조 감소하며 semantic(%.2f) 수준으로 수렴." % (cos[7], cos[0]))
    print("  => 깊은 acoustic 도 semantic 만큼 role-분화. 공유의 실질 시험대는")
    print("     semantic 에 국한되지 않는다.")